# Guided Marketing Experimentation Analysis

This notebook follows the Harbor & Pine split-test and factorial workflow from validation through a controlled rollout decision. All data are synthetic.

## 1. Load the reusable module code

The notebook delegates statistical and validation logic to tested source files so the same rules can be reused outside Jupyter.

In [ ]:
from pathlib import Path
import sys

root = Path.cwd().resolve()
while root.name != 'analytics-standard-framework' and root != root.parent:
    root = root.parent
module_root = root / '07_marketing_experimentation'
sys.path.insert(0, str(module_root / 'src'))

import generate_synthetic_data as generator
import validate_experiment_data as validation
import analyze_marketing_experiments as analysis

## 2. Generate and validate assignments

Validation happens before any effect is inspected. Duplicate, timing, consent, arm, missing-field, and immature-window defects are quarantined. Delivery failures remain in the intention-to-treat population.

In [ ]:
raw = generator.generate_raw_data()
quality = validation.validation_report(raw)
clean = validation.clean_experiment_data(raw)
quality, clean.shape

## 3. Confirm design sensitivity and sample ratio

The split test cannot quite detect its original +1.50 percentage-point planning target with 80% power after validation. The pooled factorial main effects and cell-versus-holdout contrasts are aligned with their declared sensitivity.

In [ ]:
results = analysis.run_analysis(raw)
results['power_plan'].round(4)

In [ ]:
results['srm'].round(4)

## 4. Interpret the split test

Use the assigned-customer denominator. Compare the conversion effect with both zero and the +1.50 percentage-point practical threshold, then review margin and guardrails.

In [ ]:
results['split_effects'].set_index('metric').round(4)

The lifecycle message improves conversion by about 1.44 percentage points with a confidence interval above zero, but the point estimate is slightly below the predeclared practical threshold. Revenue and contribution-margin estimates remain uncertain.

## 5. Estimate factorial main effects and interactions

Holm adjustment controls the five-effect confirmatory family. Main effects are marginal comparisons across the other balanced factors; interactions test whether one factor's effect changes with another.

In [ ]:
results['factorial_effects'][
    ['effect', 'absolute_effect', 'ci_low', 'ci_high', 'p_value', 'adjusted_p_value', 'credible_after_holm']
].round(4)

The 10% discount has a supported positive main effect versus free shipping. Message framing, additional SMS, and the two prespecified interactions are not supported in this synthetic sample.

## 6. Compare active cells with holdout

Eight cell-versus-holdout comparisons form a separate Holm-controlled family. Conversion evidence must be interpreted together with contribution margin.

In [ ]:
columns = [
    'arm', 'control_rate', 'treatment_rate', 'absolute_effect', 'ci_low', 'ci_high',
    'adjusted_p_value', 'incremental_margin_per_customer', 'margin_ci_low', 'margin_ci_high'
]
results['factorial_cells'][columns].round(4)

## 7. Apply guardrail and decision gates

A statistically credible cell is not automatically ready for scale. Confirm that every absolute guardrail passes and that margin uncertainty is visible.

In [ ]:
recommended = results['recommended_cell']
recommended_guardrails = results['guardrails'].loc[results['guardrails']['arm'].eq(recommended)]
recommended, recommended_guardrails.round(4)

## 8. Decision

Advance urgency-led messaging with a 10% discount through email only to a controlled margin-validation stage with a retained holdout. Do not approve broad rollout: conversion evidence survives multiplicity control, but incremental-margin uncertainty still includes downside. The added SMS channel has no supported main effect and therefore does not justify additional contact pressure.

## Review prompts

- Why does the split-test confidence interval matter more than a post-hoc power calculation?
- Which effects belong to separate multiplicity families, and why?
- Why can a positive conversion result still fail the rollout gate?
- What population does the factorial experiment generalize to?
- What would you monitor and what would trigger rollback in the next stage?